<a href="https://colab.research.google.com/github/mahmarani/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmarani/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [20]:
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone https://github.com/mahmarani/flyrank-ml-internship.git
    %cd flyrank-ml-internship
import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.inspection import permutation_importance

Lane: decision-support ranking of pages that are likely to be declining, so the review queue starts with the right pages.

Target (from the starter pipeline): is_declining = trend_direction == "down".

Methods: Logistic Regression (readable coefficients) and Random Forest (captures non-linear patterns and interactions). Both are compared to my Week-4 rule score on the same test pages. I use Precision@k because only the top of the queue gets reviewed, and I do not pick a model just because it is complex.

Leakage control: trend_pct, trend_direction and all *_last_30d / *_prev_30d columns are excluded because the label is built from them. Note: the 90-day columns overlap with the last 30 days, so some weak overlap remains; results are directional.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [21]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[df.impressions_90d > 0].drop_duplicates("content_id").reset_index(drop=True)
df["y"] = (df.trend_direction.str.lower() == "down").astype(int)

num = ["search_volume","competition","cpc","word_count","char_count","impressions_90d","clicks_90d",
       "sessions_90d","engaged_sessions_90d","ai_sessions_90d","days_with_impressions","days_with_sessions",
       "content_age_days","days_since_last_update","ctr","avg_position","engagement_rate","scroll_rate","ai_traffic_pct"]
cat = ["competition_level","content_type","main_intent","freshness_tier","position_tier","impression_tier"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(df, df.y, groups=df.client_id))
train, test = df.iloc[tr_idx].copy(), df.iloc[te_idx].copy()

assert set(train.client_id).isdisjoint(set(test.client_id))
print("train/test rows:", len(train), len(test))
print("train/test clients:", train.client_id.nunique(), test.client_id.nunique())
print("declining rate train/test:", round(train.y.mean(),3), round(test.y.mean(),3))

train/test rows: 22885 7115
train/test clients: 24 8
declining rate train/test: 0.55 0.517


Split: grouped by client_id (75% / 25%), so no client appears in both train and test. A random row split would let the model see other pages from the same client and look better than it really is. The split is not time-aware, because the label is built from one window, so this is a cross-client check only. The same test pages are used for the baseline and for both models.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [22]:
from sklearn.base import clone

def precision_at_k(y, score, k):
    order = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y)[order].mean())

def baseline_score(f):   # the exact ML-07 rule
    ctr_gap = (f.groupby("position_tier")["ctr"].transform("median") - f["ctr"]).clip(lower=0)
    return (0.15*f.days_since_last_update.rank(pct=True)
            + 0.45*f.impressions_90d.rank(pct=True)
            + 0.40*ctr_gap.rank(pct=True))

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
lr = Pipeline([("pre", clone(pre)), ("m", LogisticRegression(max_iter=1000))])
rf = Pipeline([("pre", clone(pre)), ("m", RandomForestClassifier(n_estimators=200, max_depth=8,
                                                                   min_samples_leaf=20, n_jobs=-1, random_state=42))])
X_cols = num + cat
lr.fit(train[X_cols], train.y)
rf.fit(train[X_cols], train.y)

scores = {"Baseline rule (ML-07)": baseline_score(test).values,
          "Logistic Regression": lr.predict_proba(test[X_cols])[:, 1],
          "Random Forest": rf.predict_proba(test[X_cols])[:, 1]}
rows = []
for name, s in scores.items():
    rows.append({"model": name,
                 "P@20": precision_at_k(test.y, s, 20), "P@50": precision_at_k(test.y, s, 50),
                 "P@100": precision_at_k(test.y, s, 100),
                 "avg_precision": average_precision_score(test.y, s)})
rows.append({"model": "Random order (base rate)", "P@20": test.y.mean(), "P@50": test.y.mean(),
             "P@100": test.y.mean(), "avg_precision": test.y.mean()})
results = pd.DataFrame(rows).round(3)
results

,model,P@20,P@50,P@100,avg_precision
0,Baseline rule (ML-07),0.800,0.840,0.630,0.517
1,Logistic Regression,0.850,0.700,0.710,0.594
2,Random Forest,0.500,0.540,0.570,0.597
3,Random order (base rate),0.517,0.517,0.517,0.517


Same test pages (8 held-out clients), same metrics (Precision@k, average precision).

Observed: the Week-4 rule has the best top-of-queue precision (P@50 = 0.84), ahead of Logistic Regression (0.70) and Random Forest (0.54, close to the 0.517 base rate). Over the whole ranking, both models beat the rule (average precision 0.59-0.60 vs 0.517, which equals random order). So the rule is good at picking a few obvious pages at the very top, while Logistic Regression ranks the full list better. Random Forest does not justify its extra complexity here. With only 8 test clients, these differences are directional, not conclusive. Decision-support only.

In [23]:
from sklearn.base import clone
res = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(df, df.y, groups=df.client_id))
    a, b = df.iloc[tr], df.iloc[te]
    m = clone(lr).fit(a[X_cols], a.y)
    for name, s in [("Baseline", baseline_score(b).values), ("LogReg", m.predict_proba(b[X_cols])[:,1])]:
        res.append({"seed": seed, "model": name, "P@50": precision_at_k(b.y, s, 50),
                    "AP": average_precision_score(b.y, s)})
pd.DataFrame(res).groupby("model")[["P@50", "AP"]].agg(["mean", "std"]).round(3)

P@50            AP       
           mean    std   mean    std
model                               
Baseline  0.668  0.120  0.620  0.091
LogReg    0.770  0.089  0.647  0.072

Robustness check: I repeated the grouped split with 10 different random seeds (same 75/25 client split each time) for the baseline rule and Logistic Regression. Mean P@50: 0.77 (std 0.09) for Logistic Regression vs 0.67 (std 0.12) for the rule. Mean average precision: 0.65 (std 0.07) vs 0.62 (std 0.09). Logistic Regression is ahead on both metrics, but the gaps are about one standard deviation, so the improvement is modest and directional. The Week-4 rule's strong P@50 (0.84) on the first split was above its 10-split average, so I do not rely on that single split. The splits overlap in clients, so they are not fully independent. Decision-support only.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [24]:
pi = permutation_importance(rf, test[X_cols], test.y, scoring="average_precision",
                            n_repeats=5, random_state=42, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X_cols).sort_values(ascending=False)
print("Top 10 features by permutation importance (Random Forest):")
print(imp.head(10).round(4))

test["rf_score"] = scores["Random Forest"]
top50 = test.sort_values("rf_score", ascending=False).head(50)
fp = top50[top50.y == 0]
print("\nFalse positives in top-50:", len(fp), "of 50")
print(fp[["avg_position", "impressions_90d", "ctr", "days_since_last_update", "rf_score"]].describe().round(2))

Top 10 features by permutation importance (Random Forest):
days_with_impressions    0.0238
content_age_days         0.0104
avg_position             0.0084
impressions_90d          0.0044
position_tier            0.0036
ctr                      0.0035
clicks_90d               0.0027
scroll_rate              0.0019
impression_tier          0.0014
days_with_sessions       0.0013
dtype: float64

False positives in top-50: 23 of 50
       avg_position  impressions_90d    ctr  days_since_last_update  rf_score
count         23.00            23.00  23.00                   23.00     23.00
mean          19.21          1314.91   0.04                   84.96      0.82
std           10.66          1502.38   0.07                   35.09      0.01
min            7.00            73.00   0.00                   20.00      0.80
25%           10.85           347.50   0.00                   97.00      0.81
50%           16.10           776.00   0.00                  104.00      0.82
75%           27.20    

Permutation importance (Random Forest, held-out clients): the top features are days_with_impressions (0.024), content_age_days (0.010) and avg_position (0.008). All importances are small, so no single feature drives the model. days_with_impressions is the number of days a page appeared in search; it likely reflects visibility over the same period the label is built from, so it may act partly as a proxy for the label rather than a cause of decline. I treat it as a directional signal, not an explanation.

Errors: 23 of the Random Forest's top-50 pages are not declining. These false positives have a worse average position (mean 19.2, median 16.1), modest impressions (median 776) and CTR close to zero (median 0.00).. The model also gives them nearly the same score (0.80-0.84), so it cannot separate them from true declines at the top. This matches its weak P@50 (0.54). The label measures only a 30-day change, so some of these pages may simply be stable and noisy. Overall, Logistic Regression is the better choice: it ranks better across the whole list (average precision 0.59-0.65) and is easier to explain. Directional, decision-support only.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.